# 01 — Ingesta de datos en Bronze

## Objetivo

Implementar la primera capa de la arquitectura Medallion
utilizando los nueve archivos CSV del dataset Olist.

Los datos serán almacenados en tablas Delta dentro de
Unity Catalog, preservando los registros originales.

## Criterios de diseño

- Conservar todas las filas de los archivos de origen.
- No eliminar duplicados ni corregir valores faltantes.
- Leer inicialmente las columnas como texto.
- Incorporar metadatos de trazabilidad.
- Implementar una carga idempotente.
- Validar la cantidad de registros de origen y destino.

La normalización, deduplicación y aplicación de reglas
de calidad se realizarán posteriormente en Silver.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType
from datetime import datetime, timezone
import hashlib
import uuid

# True: acepta de nuevo el mismo CSV (Bronze acumulativo, comportamiento solicitado).
# False: evita volver a cargar un archivo con el mismo hash de contenido registrado.
# En un Job rutinario podés sobrescribir este widget con false.
dbutils.widgets.dropdown("force_reload", "true", ["true", "false"], "Aceptar recarga repetida")
FORCE_RELOAD = dbutils.widgets.get("force_reload").lower() == "true"
RAW_PATH = "/Volumes/workspace/olist_lakehouse/raw_data/"
CATALOG = "workspace"
SCHEMA = "olist_lakehouse"
TABLAS_BRONZE = {
    "olist_customers_dataset.csv": "bronze_customers",
    "olist_geolocation_dataset.csv": "bronze_geolocation",
    "olist_order_items_dataset.csv": "bronze_order_items",
    "olist_order_payments_dataset.csv": "bronze_order_payments",
    "olist_order_reviews_dataset.csv": "bronze_order_reviews",
    "olist_orders_dataset.csv": "bronze_orders",
    "olist_products_dataset.csv": "bronze_products",
    "olist_sellers_dataset.csv": "bronze_sellers",
    "product_category_name_translation.csv": "bronze_category_translation"
}
print("Ruta de origen:", RAW_PATH)
print("Catálogo/esquema:", CATALOG, SCHEMA)
print("Fuentes configuradas:", len(TABLAS_BRONZE))
print("Recarga repetida:", FORCE_RELOAD)


In [0]:

# Diccionario para almacenar los DataFrames originales
bronze_datasets = {}

# Resultados de la validación
resultados_lectura = []

for archivo, tabla in TABLAS_BRONZE.items():

    # Lectura sin inferencia de tipos
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(RAW_PATH + archivo)
    )

    # Guardamos el DataFrame
    bronze_datasets[tabla] = df

    # Registramos cantidad de filas y columnas
    resultados_lectura.append((
        tabla,
        archivo,
        df.count(),
        len(df.columns)
    ))

# Mostrar resultados
df_validacion = spark.createDataFrame(
    resultados_lectura,
    [
        "tabla_bronze",
        "archivo_origen",
        "cantidad_registros",
        "cantidad_columnas"
    ]
)

display(df_validacion.orderBy("tabla_bronze"))


## Control de ingestas y trazabilidad

Para implementar una arquitectura incremental y acumulativa,
se incorpora una tabla de auditoría que permite registrar
las cargas de archivos realizadas en Bronze.

Cada archivo recibido podrá identificarse mediante su ruta,
su contenido y la ejecución que lo procesó.

El objetivo es:

- Evitar reprocesamientos accidentales.
- Registrar cuándo se incorporó información.
- Identificar la tabla de destino.
- Conocer la cantidad de registros procesados.
- Registrar el estado de cada ingesta.
- Facilitar la recuperación ante errores.

La auditoría controla las ejecuciones del pipeline, pero no
elimina duplicados de negocio ni modifica los registros
originales.


In [0]:

# Tabla de auditoría de Bronze
AUDIT_TABLE = f"{CATALOG}.{SCHEMA}.bronze_ingestion_audit"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
    batch_id STRING,
    source_file STRING,
    source_path STRING,
    source_file_hash STRING,
    target_table STRING,
    records_loaded BIGINT,
    ingestion_timestamp TIMESTAMP,
    status STRING,
    error_message STRING
)
USING DELTA
""")

print("Tabla de auditoría creada o verificada:")
print(AUDIT_TABLE)


In [0]:

# Verificar estructura
spark.sql(f"DESCRIBE TABLE {AUDIT_TABLE}").show(
    truncate=False
)

# Verificar cantidad de ingestas registradas
cantidad = spark.table(AUDIT_TABLE).count()

print("Ingestas registradas:", cantidad)

display(spark.table(AUDIT_TABLE))


## Detección de archivos de origen

Se implementa una etapa de descubrimiento de archivos
disponibles en el Volume de Databricks.

El objetivo es identificar las fuentes que pueden ser
procesadas por el pipeline de ingesta Bronze.

Esta etapa no modifica los archivos originales ni
escribe registros en las tablas de destino.

La detección constituye el primer paso para implementar
un procesamiento incremental y acumulativo.


In [0]:

# Listar los archivos disponibles en el Volume
archivos_disponibles = dbutils.fs.ls(RAW_PATH)

# Identificar únicamente archivos CSV
archivos_csv = [
    archivo
    for archivo in archivos_disponibles
    if archivo.name.lower().endswith(".csv")
]

# Mostrar los archivos detectados
resultados = [
    (
        archivo.name,
        archivo.path,
        archivo.size
    )
    for archivo in archivos_csv
]

df_archivos = spark.createDataFrame(
    resultados,
    ["archivo", "ruta", "tamano_bytes"]
)

display(df_archivos.orderBy("archivo"))

print("Archivos CSV detectados:", len(archivos_csv))


In [0]:

# Validar que todos los CSV detectados tengan tabla Bronze
archivos_detectados = {
    archivo.name for archivo in archivos_csv
}

archivos_configurados = set(TABLAS_BRONZE.keys())

sin_configurar = archivos_detectados - archivos_configurados
no_encontrados = archivos_configurados - archivos_detectados

print("Archivos detectados:", len(archivos_detectados))
print("Archivos configurados:", len(archivos_configurados))
print("Sin configurar:", sin_configurar)
print("No encontrados:", no_encontrados)

if sin_configurar or no_encontrados:
    raise ValueError(
        "Existen diferencias entre los archivos "
        "detectados y la configuración Bronze."
    )

print("Validación exitosa: 9 archivos correctamente mapeados.")


## Ingesta acumulativa con linaje por fila

La única celda de escritura es la siguiente. Se elimina la carga de prueba independiente
`COPY INTO bronze_customers` para no procesar `customers` dos veces.

- Con `force_reload=true`, cada ejecución **agrega todas las filas** de los nueve CSV, aunque ya hayan llegado antes.
- Con `force_reload=false`, se omite un archivo cuyo SHA-256 conste como cargado exitosamente.
- Se mantienen las columnas de negocio originales **STRING** y se agregan cuatro columnas técnicas (`__ingest_*`).
- Silver ignora esas columnas técnicas al calcular los hashes y al publicar su grano único.

**Advertencia:** la primera ejecución tras este cambio agrega columnas técnicas a las tablas Bronze existentes. Los registros históricos conservan valores NULL en esas columnas y se tratan como lote legado.

### Control importante
Se ha retirado la segunda ingesta aislada de `customers`. Toda la carga se realiza **una sola vez por fuente y ejecución** en la siguiente celda.

In [0]:
# Carga Bronze: APPEND deliberado, sin deduplicación de negocio.
# Incluye rastreo por lote, archivo, SHA256 y timestamp por cada fila nueva.
BATCH_ID = str(uuid.uuid4())
resultados_ingesta = []
INGEST_COLS = {
    "__ingest_batch_id": "STRING",
    "__ingest_file": "STRING",
    "__ingest_file_sha256": "STRING",
    "__ingest_at": "TIMESTAMP"
}

def sha256_archivo(path, chunk_size=4 * 1024 * 1024):
    dig = hashlib.sha256()
    with open(path, "rb") as fh:
        while True:
            blob = fh.read(chunk_size)
            if not blob:
                break
            dig.update(blob)
    return dig.hexdigest()

for archivo, tabla in TABLAS_BRONZE.items():
    tabla_completa = f"{CATALOG}.{SCHEMA}.{tabla}"
    ruta_archivo = RAW_PATH + archivo
    file_hash = None
    try:
        file_hash = sha256_archivo(ruta_archivo)
        df_origen = (spark.read.option("header", True)
                     .option("inferSchema", False).option("multiLine", True)
                     .option("quote", '"').option("escape", '"')
                     .csv(ruta_archivo))
        columnas_negocio = df_origen.columns
        if any(c.startswith("__ingest_") for c in columnas_negocio):
            raise ValueError("El CSV contiene una columna reservada __ingest_*")

        columnas_sql = ",\n".join(
            f"`{c.replace('`', '``')}` STRING" for c in columnas_negocio
        )
        spark.sql(f"CREATE TABLE IF NOT EXISTS {tabla_completa} ({columnas_sql}) USING DELTA")
        columnas_existentes = set(spark.table(tabla_completa).columns)
        faltantes_negocio = sorted(set(columnas_negocio) - columnas_existentes)
        if faltantes_negocio:
            raise ValueError(f"Columnas de negocio faltantes en {tabla_completa}: {faltantes_negocio}")
        for columna, tipo in INGEST_COLS.items():
            if columna not in columnas_existentes:
                spark.sql(f"ALTER TABLE {tabla_completa} ADD COLUMNS (`{columna}` {tipo})")

        # En modo normal optativo se evita una segunda carga del mismo contenido.
        # Esta comprobación NO deduplica filas: es una política de ingesta de archivos.
        ya_ingestado = (spark.table(AUDIT_TABLE)
            .filter((F.col("source_file_hash") == file_hash)
                    & (F.col("target_table") == tabla_completa)
                    & (F.col("status") == "SUCCESS")
                    & (F.col("records_loaded") > 0))
            .limit(1).count() > 0)
        if not FORCE_RELOAD and ya_ingestado:
            insertados, estado = 0, "SKIPPED"
        else:
            insertados = int(df_origen.count())
            nuevo = (df_origen
                .withColumn("__ingest_batch_id", F.lit(BATCH_ID))
                .withColumn("__ingest_file", F.lit(archivo))
                .withColumn("__ingest_file_sha256", F.lit(file_hash))
                .withColumn("__ingest_at", F.current_timestamp()))
            nuevo.write.format("delta").mode("append").saveAsTable(tabla_completa)
            estado = "SUCCESS"
        resultados_ingesta.append((tabla, archivo, insertados, estado, None, file_hash))
        print(f"{estado} | {tabla}: {insertados:,} registros recibidos")
    except Exception as exc:
        resultados_ingesta.append((tabla, archivo, 0, "ERROR", str(exc)[:1500], file_hash))
        print(f"ERROR | {tabla}: {exc}")

# Esquema explícito: evita CANNOT_DETERMINE_TYPE cuando todos los errores son NULL.
schema_resultados = StructType([
    StructField("tabla_bronze", StringType(), False),
    StructField("archivo_origen", StringType(), False),
    StructField("filas_insertadas", LongType(), False),
    StructField("estado", StringType(), False),
    StructField("error", StringType(), True),
    StructField("source_file_hash", StringType(), True)
])
df_resultados = spark.createDataFrame(resultados_ingesta, schema_resultados)
display(df_resultados.orderBy("tabla_bronze"))
print("BRONZE BATCH_ID:", BATCH_ID)
# Si la tarea se llama Bronze en el Job, la salida puede enlazarse con
# {{tasks.Bronze.values.batch_id}}. El resto del pipeline también puede
# descubrir el último lote en bronze_ingestion_audit.
try:
    dbutils.jobs.taskValues.set(key="batch_id", value=BATCH_ID)
except Exception as exc:
    print("Task value no disponible fuera de Jobs:", str(exc)[:160])


In [0]:
# Resultado resumido, compatible con el Job y con tablas sin altas.
display(df_resultados.select("tabla_bronze", "filas_insertadas", "estado", "error")
        .orderBy("tabla_bronze"))
print("Registros ingresados en el lote:", sum(r[2] for r in resultados_ingesta))
print("Tablas con error:", sum(r[3] == "ERROR" for r in resultados_ingesta))


In [0]:
# Persistir auditoría de cada fuente, incluso si otra falló.
# Conservamos la estructura original de bronze_ingestion_audit.
AUDIT_TABLE = f"{CATALOG}.{SCHEMA}.bronze_ingestion_audit"
audit_schema = StructType([
    StructField("batch_id", StringType()),
    StructField("source_file", StringType()),
    StructField("source_path", StringType()),
    StructField("source_file_hash", StringType()),
    StructField("target_table", StringType()),
    StructField("records_loaded", LongType()),
    StructField("ingestion_timestamp", TimestampType()),
    StructField("status", StringType()),
    StructField("error_message", StringType())
])
marca_tiempo = datetime.now(timezone.utc)
audit_rows = [(
    BATCH_ID, archivo, RAW_PATH + archivo, file_hash,
    f"{CATALOG}.{SCHEMA}.{tabla}", int(insertados), marca_tiempo, estado, error
) for tabla, archivo, insertados, estado, error, file_hash in resultados_ingesta]
spark.createDataFrame(audit_rows, audit_schema).write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)

# La capa Bronze es acumulativa; el destino debe contener AL MENOS el lote de origen.
validaciones = []
for archivo, tabla in TABLAS_BRONZE.items():
    origen = int(bronze_datasets[tabla].count())
    acumulado = int(spark.table(f"{CATALOG}.{SCHEMA}.{tabla}").count())
    valida = acumulado >= origen
    validaciones.append((tabla, origen, acumulado, "OK" if valida else "REVISAR"))
schema_validaciones = StructType([
    StructField("tabla", StringType()), StructField("registros_origen", LongType()),
    StructField("registros_bronze", LongType()), StructField("estado", StringType())
])
display(spark.createDataFrame(validaciones, schema_validaciones).orderBy("tabla"))
if not all(r[3] == "OK" for r in validaciones):
    raise AssertionError("Bronze contiene menos filas que el archivo fuente en alguna entidad")
errores = [r for r in resultados_ingesta if r[3] == "ERROR"]
if errores:
    raise RuntimeError(f"Falló la ingesta de {len(errores)} fuente(s). Revisá la auditoría Bronze.")
print("TOTAL BRONZE ACUMULADO:", sum(r[2] for r in validaciones))
print("BRONZE EXITOSO | BATCH_ID:", BATCH_ID)
